# PHASE 6: FUSION BRANCH - SEQUENTIAL STRATEGY COMPARISON

**Objectif:** Comparer 5 stratégies de fusion sur Part B (données non vues) en exécution séquentielle.

**Architecture:**
```
Part B (31.8k samples) - UNSEEN DATA
      ↓
  Split 50/50
 /            \
Fusion Train   Fusion Test
(15.9k)        (15.9k)
   ↓              ↓
Optimize (1s   Evaluate (1s
à la fois)     à la fois)
```

**Stratégies testées (séquentiellement):**
1. **Cascading**: If Style confiant → Style, else Knowledge
2. **Confidence-Weighted**: Poids fixes (0.92 vs 0.32)
3. **Disagreement-Adaptive**: Adaptation selon accord/désaccord
4. **Weighted + Threshold** ⭐: Gridsearch 125 configs
5. **Stacked RF** ⭐: Meta-learner RandomForest

**Résultats attendus:**
- Style baseline: 92% F1
- Knowledge baseline: 32% F1
- Fusion: 88-91% F1


In [ ]:
import sys
from pathlib import Path

# Setup path for gpu_utils
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import GPU utilities
from gpu_utils import setup_training_device, print_device_info, get_device_info

# Configure GPU/Device for all inference/fusion
device = setup_training_device(verbose=False)
print_device_info("🔀 Fusion Branch - GPU Configuration")

# Verify CUDA is working
import torch
print(f"\n📋 PyTorch Status:")
print(f"   Version: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   ✅ Ready for fusion inference on GPU!")
    print(f"   ⏱️  Expected time: 10-20 minutes (5 strategies)")
else:
    print(f"   ⚠️  Using CPU (still fast for fusion - no training)")

print("\n" + "="*60)
print("✅ GPU Setup Complete - Fusion will use GPU for inference")
print("="*60 + "\n")

# Fusion Branch - Simplified Pipeline

This notebook runs the complete fusion pipeline by calling individual Python scripts.
Each cell is self-contained and executes a specific phase of the fusion analysis.

**Pipeline Overview:**
1. Verify models exist
2. Load predictions from frozen models
3. Split Part B into train/test
4-8. Run 5 fusion strategies
9. Compare and visualize results

date: 2026-04-12

## Section 0: Verify Frozen Models

In [46]:
import subprocess
from pathlib import Path
import sys

# Change to fusion_branch and run script
%cd fusion_branch
!python 00_verify_models.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
Section 1: VÉRIFICATION MODÈLES GELÉS (Part A Training)
✅ Style best_model              : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch/results/best_model.pkl
✅ Style RoBERTa                 : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch/roberta_fine_tunned
✅ Knowledge Claim Detector      : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/claim_detector_model
✅ Knowledge NLI                 : /home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/knowledge_branch/my_claim_model
✅ TOUS LES MODÈLES PRÉSENTS - Prêt pour Phase 6B/6C

✅ Dossier results créé pour stockage résultats
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 1: Load Predictions from Frozen Models

In [47]:
%cd fusion_branch
!python 01_load_predictions.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 2: CHARGER PART B ET GÉNÉRER PRÉDICTIONS BRUTES
❌ Error executing evidence_loader: No module named 'config'
Traceback (most recent call last):
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/01_load_predictions.py", line 20, in <module>
    spec.loader.exec_module(evidence_loader)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap_external>", line 1023, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/evidence_loader.py", line 10, in <module>
    from config import PATHS
ModuleNotFoundError: No module named 'config'

✅ Prédictions chargées:
   - Style predictions: 31804 samples
   - Knowledge predictions: 31804 samples
   - Ground truth labels: 31804 samples
   - Classe 0 (FAKE

## Section 2: Split Part B Data

In [48]:
%cd fusion_branch
!python 02_split_data.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 3: SPLIT PART B EN FUSION_TRAIN/TEST (50/50)

📊 Split Results:
   Fusion Train: 15902 samples (Classe 0: 5968, Classe 1: 9934)
   Fusion Test:  15902 samples (Classe 0: 5969, Classe 1: 9933)

✅ Données de fusion sauvegardées:
   - fusion_train.csv: 15902 rows
   - fusion_test.csv: 15902 rows
   - results/part_b_split.pkl: données splits pour stratégies
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 3: Strategy 1 - Cascading (Style First)

In [49]:
%cd fusion_branch
!python 03_strategy_1.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 4: STRATÉGIE 1 - CASCADING (Style First)

🔍 Gridsearch sur fusion_train...
✅ Meilleur seuil trouvé: 0.50 (F1 train: 0.6013)

📊 Résultats Stratégie 1 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_1_cascading_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 4: Strategy 2 - Confidence-Weighted Voting

In [50]:
%cd fusion_branch
!python 04_strategy_2.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
Traceback (most recent call last):
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/04_strategy_2.py", line 12, in <module>
    from strategy_2_confidence_weighted import ConfidenceWeightedVoting
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/strategy_2_confidence_weighted.py", line 15, in <module>
    from config import STYLE_BASELINE_F1, KNOWLEDGE_BASELINE_F1
ModuleNotFoundError: No module named 'config'
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 5: Strategy 3 - Disagreement-Adaptive Weighting

In [51]:
%cd fusion_branch
!python 05_strategy_3.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 6: STRATÉGIE 3 - DISAGREEMENT-ADAPTIVE WEIGHTING

🔍 Gridsearch sur fusion_train...
✅ Meilleur poids trouvé: 0.30 (F1 train: 0.6013)

📊 Résultats Stratégie 3 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_3_disagreement_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 6: Strategy 4 - Weighted Voting + Threshold ⭐

In [52]:
%cd fusion_branch
!python 06_strategy_4.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 7: STRATÉGIE 4 - WEIGHTED + THRESHOLD ⭐

🔍 Gridsearch sur fusion_train (125 configurations)...
✅ Configurations testées: 125
✅ Meilleurs paramètres trouvés:
   w_style=0.50, w_knowledge=0.45, threshold=0.40
   F1 train: 0.6013

📊 Résultats Stratégie 4 (Test unseen):
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035
✅ Résultats sauvegardés: strategy_4_weighted_threshold_report.json
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 7: Strategy 5 - Stacked RandomForest ⭐

In [53]:
%cd fusion_branch
!python 07_strategy_5.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch
Traceback (most recent call last):
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/07_strategy_5.py", line 13, in <module>
    from strategy_5_stacked_rf import StackedRandomForestFusion
  File "/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch/strategy_5_stacked_rf.py", line 16, in <module>
    from config import RANDOM_STATE
ModuleNotFoundError: No module named 'config'
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


## Section 8: Final Comparison & Visualization

In [54]:
%cd fusion_branch
!python 08_comparison_visualize.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/fusion_branch

Section 9-10: COMPARAISON FINALE & VISUALISATION

Évaluation des BASELINES (Style & Knowledge seuls)

✅ Style (baseline):
   Accuracy:  0.3054
   Precision: 0.4219
   Recall:    0.3027
   F1-Score:  0.3525

✅ Knowledge (baseline):
   Accuracy:  0.5048
   Precision: 0.6294
   Recall:    0.5041
   F1-Score:  0.5598

Évaluation des STRATÉGIES DE FUSION

✅ Strategy 1: Cascading:
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035

✅ Strategy 2: Conf-Weighted:
   Accuracy:  0.4357
   Precision: 0.5500
   Recall:    0.5309
   F1-Score:  0.5403

✅ Strategy 3: Disagreement:
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035

✅ Strategy 4: Weighted+Thresh:
   Accuracy:  0.4654
   Precision: 0.5622
   Recall:    0.6515
   F1-Score:  0.6035

✅ Strategy 5: Stacked RF ⭐:
   Accuracy:  0.7706
   Precision: 0.7350
   Recall:    0.9894
   F1-Score:  0

---

## Section 10: Visualisation & Résumé Final

Graphiques de comparaison des 5 stratégies.


In [55]:
print("\n✅ Les résultats et visualisations sont générés dans fusion_branch/results/")
print("   Consultez:")
print("   - comparison_table.csv")
print("   - best_strategy_analysis.txt")
print("   - fusion_strategy_comparison.png")


✅ Les résultats et visualisations sont générés dans fusion_branch/results/
   Consultez:
   - comparison_table.csv
   - best_strategy_analysis.txt
   - fusion_strategy_comparison.png
